# Lewisham · records whose site is outside Lewisham

**EDA exception to the golden rule, approved by the user on 2026-11-11.** Some Lewisham
records (borough Lewisham) describe sites in other boroughs. To confirm what they are,
this notebook reads a few records of **other** boroughs. Limits: read-only, only the
references and addresses being checked, only the fields needed to compare; nothing from
here goes into the bot, `semantic/`, tests or fixtures; only conclusions are written down.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [ ]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Do the other boroughs have the original applications?

1. Two Lewisham records quote the other borough's reference in their description:
   DC/26/142577 (Bromley, "their ref: 23/00178/S73A") and DC/24/134407 (Tower Hamlets,
   "PA/21/02377"). Look those references up in Bromley's and Tower Hamlets' records.
2. Four Lewisham records with no description and an address outside Lewisham
   (description notebook, Step 2): look for the same street in that borough's records,
   decided around the same time.

Only `id`, `lpa_name`, `borough`, `lpa_app_no`, `site_name`, `decision`, `decision_date`
and the first 80 characters of `description` are read.

In [ ]:
OUT = ["id", "lpa_name", "borough", "lpa_app_no", "site_name", "decision", "decision_date", "description"]


def outside(authority: str, must: list[dict]) -> pd.DataFrame:
    # Read-only EDA lookup in another authority's records (approved exception, see top).
    body = {"size": 20, "_source": OUT,
            "query": {"bool": {"filter": [{"term": {"lpa_name.raw": authority}}, *must]}}}
    found = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=OUT)
    found["description"] = found["description"].str.slice(0, 80)
    return found


lpas = [b["key"] for b in search({"size": 0, "aggs": {"lpa": {"terms": {"field": "lpa_name.raw", "size": 100}}}})["aggregations"]["lpa"]["buckets"]]
print("Neighbours present as authorities:", [n for n in lpas if n in ["Bromley", "Southwark", "Tower Hamlets", "Greenwich"]])

In [ ]:
# 1. References quoted in Lewisham descriptions.
pd.set_option("display.max_colwidth", 80)
pd.concat([
    outside("Bromley", [{"match_phrase": {"lpa_app_no": "23/00178/S73A"}}]),
    outside("Tower Hamlets", [{"match_phrase": {"lpa_app_no": "PA/21/02377"}}]),
])

In [ ]:
# 2. Addresses outside Lewisham on Lewisham records with no description.
CHECKS = [
    ("Lewisham-DC_19_114630", "Bromley", "HILLCREST ROAD", "19/11/2019"),
    ("Lewisham-DC_19_114677", "Bromley", "LYDSTEP ROAD", "20/11/2019"),
    ("Lewisham-DC_19_115107", "Bromley", "CRYSTAL PALACE PARK ROAD", "06/01/2020"),
    ("Lewisham-DC_20_116629", "Southwark", "BUSH ROAD", "17/12/2020"),
]
for lewisham_id, authority, street, lewisham_date in CHECKS:
    d = pd.to_datetime(lewisham_date, format="%d/%m/%Y")
    window = {"range": {"decision_date": {"gte": (d - pd.Timedelta(days=365)).strftime("%d/%m/%Y"),
                                          "lte": (d + pd.Timedelta(days=365)).strftime("%d/%m/%Y"),
                                          "format": "dd/MM/yyyy"}}}
    found = outside(authority, [{"match_phrase": {"site_name": street}}, window])
    print(f"{lewisham_id} (Lewisham decision {lewisham_date}) -> {authority}, {street}, decided within a year: {len(found)}")
    if len(found):
        print(found.drop(columns=["borough"]).to_string(index=False))
    print()

In [ ]:
# 3. Follow-ups: Bromley's own OBS record at 8 Lydstep Road (full description), the
# application it names (19/3662/HD) in Greenwich, and Bromley's 23/00178 by number only.
lydstep = search({"size": 1, "_source": ["id", "description"],
                  "query": {"bool": {"filter": [{"term": {"lpa_name.raw": "Bromley"}}, {"term": {"id": "Bromley-19_04705_ADJ"}}]}}})
print(lydstep["hits"]["hits"][0]["_source"]["description"][:300])
print()
print(outside("Greenwich", [{"match_phrase": {"lpa_app_no": "19/3662/HD"}}]).drop(columns=["borough"]).to_string(index=False))
print()
print(outside("Bromley", [{"match_phrase": {"lpa_app_no": "23/00178"}}]).drop(columns=["borough"]).to_string(index=False))

### Findings · Step 1

**Confirmed: the original applications are in the other boroughs' records.**

| Lewisham record | Lewisham decision | Original application found |
|---|---|---|
| DC/24/134407 (quotes PA/21/02377) | none (portal: removed) | Tower Hamlets PA/21/02377/A1, Aberfeldy Estate, "Referred to Mayor" 03/06/2024 |
| DC/26/142577 (quotes Bromley 23/00178/S73A) | none (portal: removed) | Bromley 23/00178/FULL1, Blenheim Shopping Centre (the S73A variation itself not found) |
| DC/19/114677, 8 Lydstep Road, Chislehurst BR7 | Comment Issued, 20/11/2019 | **Greenwich** 19/3662/HD, same address, Approved 12/12/2019 |

- **8 Lydstep Road is a Greenwich application, not Bromley:** the postcode (BR7) and town
  (Chislehurst) pointed to Bromley, wrongly. **A postcode or town in `site_name` does
  not tell the borough.**
- **Neighbours were consulted:** Bromley also has a record for 8 Lydstep Road,
  19/04705/ADJ, decision "No Objection to Proposal (OBS only)", description "…(19/3662/HD)
  (CONSULTATION BY ROYAL BOROUGH OF GREENWICH COUNCIL)". So the Datahub records a
  consultee's response as the consultee's own record, with the consultee as borough. Lewisham's
  DC/19/114677 is the same kind of record.
- **"OBS" fits observations by a consulted neighbour:** Bromley's consultation record uses
  the same OBS-only value. Still an inference, but now from the Datahub's own text.
- **Not matched:** 73 Hillcrest Road (Bromley) — nothing within a year; 94 Crystal Palace
  Park Road — only an unrelated tree works record at no. 13; 19 Bush Road (Southwark) —
  Southwark has many Bush Road records (Haddonfield Estate, SE8), but no 19 Bush Road.
- **For Lewisham:** these consultation records are not Lewisham applications. Their sites
  are outside Lewisham and the application itself is counted in the other borough.

## Step 2 · Which Lewisham records are consultation responses?

Decided with the user (2026-11-11): consultation responses are dropped at load. This step
looks for a condition that catches them and nothing else. For every Lewisham record with
an opinion-type decision, split by `lpa_app_no` prefix, count how many descriptions:
- mention **consultation or observations** ("consultation", "consulted", "observation"),
- **name another authority** (a neighbouring borough, "Royal Borough", "Mayor of London",
  "adjoining"),
- are **missing**.

In [ ]:
LCOLS = ["id", "lpa_app_no", "decision", "status", "site_name", "description"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": LCOLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

lew = pd.DataFrame(rows).reindex(columns=LCOLS)
lew["prefix"] = lew["lpa_app_no"].str.extract(r"^([A-Za-z]*)", expand=False)
OPINION = ["No Objection to Proposal (OBS only)", "Objection Raised to Proposal (OBS only)", "Comment Issued", "Closed"]
text = lew["description"].fillna("").str.lower()
lew["says_consultation"] = text.str.contains(r"consult|observation")
lew["names_other_authority"] = text.str.contains(
    r"bromley|southwark|greenwich|tower hamlets|lambeth|royal borough|mayor of london|adjoining|neighbouring borough")
lew["no_description"] = lew["description"].isna()

op = lew[lew["decision"].isin(OPINION)]
summary = op.groupby(["decision", "prefix"]).agg(
    records=("id", "size"), says_consultation=("says_consultation", "sum"),
    names_other_authority=("names_other_authority", "sum"), no_description=("no_description", "sum"))
summary

In [ ]:
# The same text markers on records that are NOT opinion-type, to see whether they also
# catch ordinary Lewisham applications.
rest = lew[~lew["decision"].isin(OPINION)]
print("Other records:", len(rest), "| say consultation/observation:", int(rest["says_consultation"].sum()),
      "| name another authority:", int(rest["names_other_authority"].sum()))
print(rest.loc[rest["says_consultation"] | rest["names_other_authority"], "decision"].fillna("(none)").value_counts().to_string())

In [ ]:
# A sample of OBS-only DC/ records whose description shows neither marker.
pd.set_option("display.max_colwidth", 90)
obs_dc = op[op["decision"].str.contains("OBS only") & (op["prefix"] == "DC")]
sample = obs_dc[~obs_dc["says_consultation"] & ~obs_dc["names_other_authority"] & ~obs_dc["no_description"]]
print(len(sample), "OBS-only DC/ records without a marker; first 15:")
sample[["id", "decision", "site_name", "description"]].head(15)

### Findings · Step 2

**No simple condition separates consultation responses.**

- **OBS-only values are not only consultations.** Many OBS-only DC/ records are Lewisham's
  own opinions on sites **in Lewisham**: EIA Screening and Scoping Opinions (Leegate
  Shopping Centre SE12, Ladywell Leisure Centre SE13, SELCHP SE14, Trundley's Road SE8, …)
  and requests for extended construction hours (Loampit Vale SE13, St Dunstan's College
  SE6, Ashmead Primary SE8). Dropping by decision value would drop these.
- **Some are clearly elsewhere:** Eltham College SE9 (site_name says Bromley), 81 Newgate
  Street EC1A (City of London), North Dene, Beckenham Place Park BR3.
- **Text markers don't work either:** "consult/observation" or a named authority appear in
  only 58 / 50 of the 431 opinion-type records, and also in ordinary records (34 and 438,
  mostly Approved / Refused: e.g. "adjoining" property).
- **PRE/ records never show a marker** (0 of 35).
- So the opinion-type values mean "the council gave an opinion", about Lewisham sites as
  well as neighbours' applications. Which records are consultations on another borough's
  application has to be confirmed per record.

## Step 3 · Cross-borough check of the opinion-type DC/ records

Decided with the user (2026-11-11): consultation responses are dropped **by id**, and only
when confirmed; Lewisham's own records are never dropped. Candidates: every Lewisham DC/
record with an opinion-type decision. For each, search **every other authority** for a
record decided within a year (before or after) at the same site:
- **postcode:** the full postcode in the Lewisham `site_name`, found in the other
  record's `site_name`; or, without a postcode,
- **address:** the first part of the Lewisham `site_name` (before the first comma, e.g.
  "8 LYDSTEP ROAD"), if it starts with a house number.

Only the other record's id, authority, reference, decision, decision_date and the start of
its site_name are kept, for review. Nothing is dropped from this step alone.

In [ ]:
import re

POSTCODE = re.compile(r"\b([A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})\b")
cand = lew[lew["decision"].isin(OPINION) & (lew["prefix"] == "DC")].copy()
dates = search({"size": 1000, "_source": ["id", "decision_date"],
                "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": cand["id"].tolist()}}]}}})
cand = cand.merge(pd.DataFrame([h["_source"] for h in dates["hits"]["hits"]]), on="id", how="left")


def site_keys(site: str | None) -> tuple[str | None, str | None]:
    site = (site or "").upper()
    postcode = POSTCODE.search(site)
    first = site.split(",")[0].strip()
    address = first if re.match(r"^\d", first) else None
    return (postcode.group(1) if postcode else None), address


def other_authorities(postcode: str | None, address: str | None, decided: str | None) -> list[dict]:
    # Read-only EDA lookup across other authorities (approved exception, see top).
    if not decided or not (postcode or address):
        return []
    d = pd.to_datetime(decided, format="%d/%m/%Y")
    window = {"range": {"decision_date": {"gte": (d - pd.Timedelta(days=365)).strftime("%d/%m/%Y"),
                                          "lte": (d + pd.Timedelta(days=365)).strftime("%d/%m/%Y"),
                                          "format": "dd/MM/yyyy"}}}
    phrase = {"match_phrase": {"site_name": postcode or address}}
    body = {"size": 5, "_source": ["id", "lpa_name", "lpa_app_no", "decision", "decision_date", "site_name"],
            "query": {"bool": {"filter": [window, phrase],
                               "must_not": [{"term": {"lpa_name.raw": "Lewisham"}}]}}}
    return [h["_source"] for h in search(body)["hits"]["hits"]]


found = []
for row in cand.itertuples():
    postcode, address = site_keys(row.site_name)
    for hit in other_authorities(postcode, address, row.decision_date):
        found.append({"lewisham_id": row.id, "lewisham_decision": row.decision, "lewisham_date": row.decision_date,
                      "match_on": "postcode" if postcode else "address", "key": postcode or address,
                      "other_authority": hit.get("lpa_name"), "other_ref": hit.get("lpa_app_no"),
                      "other_decision": hit.get("decision"), "other_date": hit.get("decision_date"),
                      "other_site": (hit.get("site_name") or "").replace("\n", " ").replace("\r", " ")[:45]})
matches = pd.DataFrame(found)
no_key = cand["site_name"].map(lambda s: site_keys(s) == (None, None)).sum()
print("Candidates:", len(cand), "| without a postcode or numbered address:", int(no_key),
      "| with at least one match:", matches["lewisham_id"].nunique() if len(matches) else 0)
print(matches.groupby("other_authority")["lewisham_id"].nunique().sort_values(ascending=False).to_string() if len(matches) else "")

In [ ]:
# Review: is the other record a real decision (the application) or itself an opinion
# (another consultee)? By match type.
matches["other_is_opinion"] = matches["other_decision"].isin(OPINION)
print(pd.crosstab(matches["match_on"], matches["other_is_opinion"]).to_string())
print()
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 300)
print(matches.drop(columns=["other_is_opinion"]).sort_values(["other_authority", "lewisham_id"]).to_string(index=False))

### Findings · Step 3

396 candidates; 115 have neither a postcode nor a numbered address, so they cannot be
looked up this way. 85 have at least one match in another authority (Greenwich 34,
Southwark 26, Tower Hamlets 10, Bromley 10, others 1–4).

**A match on the site is not a confirmation.** Reviewing the 248 matched pairs:
- **Address-only matches on a bare number are noise:** a Lewisham site_name starting
  "87, …" matched five unrelated "87 …" addresses in Ealing.
- **A postcode covers several addresses:** SE3 9RT matched 45, 45A and 55A Lee Road.
- **One site has many applications:** Eltham College (SE9 4QF) matched a tree works
  application, an advertisement, conditions… not necessarily the one Lewisham commented on.
- **Other consultees match too:** some matches are themselves OBS-only or Comment Issued
  (47 of 248), e.g. Bromley and Croydon commenting on the same Crystal Palace schemes.
- **Good matches exist:** 101A Lee Road (Greenwich 19/2359/HD, Approved 27/08/2019; Lewisham
  No Objection 14/08/2019), 57 Cator Road (Bromley 21/02450/FULL1, Refused 14/09/2021;
  Lewisham Comment Issued 05/08/2021), North Dene, Beckenham Place Park (Bromley
  20/01918/FULL1, Refused 19/08/2020; Lewisham Objection Raised 02/07/2020), 1 Undershaft
  EC3A (City of London).
- **So the list of ids to drop needs a stricter match** (same house number and street, the
  other record a real decision, decided shortly after Lewisham's response, similar
  description), and a person to review the result. Not done yet.

## Step 4 · Stricter match

For each candidate from Step 3 (Lewisham DC/ record, opinion-type decision):
1. **Address:** the first part of the Lewisham `site_name` must be a house number followed
   by a street (e.g. "101A LEE ROAD"); the other record's `site_name` must contain that
   exact phrase.
2. **A real decision:** the other record's decision is not opinion-type.
3. **Timing:** the other record is decided from 14 days before to 120 days after
   Lewisham's response.
4. **Description:** word overlap (Jaccard, words of 4+ letters) between the two
   descriptions, shown; **0.2 or more** counts as similar. Without a Lewisham description
   the pair is listed but marked "no description".

The result is for a person to review; nothing is dropped from this step alone.

In [ ]:
ADDRESS = re.compile(r"^(\d+[A-Z]?(?:-\d+[A-Z]?)?\s+[A-Z][A-Z' ]+)$")


def numbered_street(site: str | None) -> str | None:
    first = (site if isinstance(site, str) else "").upper().split(",")[0].strip()
    m = ADDRESS.match(first)
    return m.group(1).strip() if m else None


def words(text: object) -> set[str]:
    return set(re.findall(r"[a-z]{4,}", text.lower())) if isinstance(text, str) else set()


def jaccard(a: set[str], b: set[str]) -> float:
    return len(a & b) / len(a | b) if a and b else 0.0


strict = []
for row in cand.itertuples():
    address = numbered_street(row.site_name)
    if not address or not isinstance(row.decision_date, str):
        continue
    d = pd.to_datetime(row.decision_date, format="%d/%m/%Y")
    window = {"range": {"decision_date": {"gte": (d - pd.Timedelta(days=14)).strftime("%d/%m/%Y"),
                                          "lte": (d + pd.Timedelta(days=120)).strftime("%d/%m/%Y"),
                                          "format": "dd/MM/yyyy"}}}
    # Read-only EDA lookup across other authorities (approved exception, see top).
    body = {"size": 10, "_source": ["id", "lpa_name", "lpa_app_no", "decision", "decision_date", "site_name", "description"],
            "query": {"bool": {"filter": [window, {"match_phrase": {"site_name": address}}],
                               "must_not": [{"term": {"lpa_name.raw": "Lewisham"}},
                                            {"terms": {"decision.raw": OPINION}}]}}}
    for h in search(body)["hits"]["hits"]:
        o = h["_source"]
        strict.append({"lewisham_id": row.id, "lewisham_decision": row.decision, "lewisham_date": row.decision_date,
                       "address": address, "other_authority": o.get("lpa_name"), "other_ref": o.get("lpa_app_no"),
                       "other_decision": o.get("decision"), "other_date": o.get("decision_date"),
                       "similarity": round(jaccard(words(row.description), words(o.get("description"))), 2),
                       "lewisham_has_description": isinstance(row.description, str)})
strict = pd.DataFrame(strict)
with_address = cand["site_name"].map(numbered_street).notna().sum()
print("Candidates with a numbered street address:", int(with_address), "| pairs found:", len(strict),
      "| Lewisham records with a pair:", strict["lewisham_id"].nunique())
strict["verdict"] = "different description"
strict.loc[strict["similarity"] >= 0.2, "verdict"] = "similar description"
strict.loc[~strict["lewisham_has_description"], "verdict"] = "no description"
print(strict.groupby("verdict")["lewisham_id"].nunique().to_string())

In [ ]:
pd.set_option("display.width", 300); pd.set_option("display.max_rows", 200)
strict.drop(columns=["lewisham_has_description"]).sort_values(["verdict", "similarity"], ascending=[True, False])

### Findings · Step 4

114 of the 396 candidates have a numbered street address; 15 of them have a strict match
(21 pairs): Greenwich 11, Southwark 3, Bromley 1.

**Confirmed (14):** the same address, a real decision by the other authority soon after,
and the same proposal:
- 13 with a similar description (word overlap 0.22–1.00): DC/19/111843 (43 Winn Road),
  DC/19/113239 (840 Old Kent Road), DC/19/113245 (101A Lee Road), DC/21/120099 (10 Evelina
  Road), DC/22/127453 (45 Lee Road), DC/22/129609 (143 Lee Road), DC/23/130171 (19-49 Bush
  Road), DC/23/131797 (89 Winn Road), DC/23/132609 and DC/23/132610 (62 Royal Hill),
  DC/24/134968 (43 Crosslet Vale), DC/25/138839 (81 Lee Road), DC/25/140753 (35 Crosslet
  Vale).
- DC/19/114677 (8 Lydstep Road): no Lewisham description, but confirmed in Step 1 (Greenwich
  19/3662/HD; Bromley's own record names the Greenwich consultation).

**For review (1):** DC/21/122776 (57 Cator Road): Bromley 21/02450/FULL1 at the same
address 40 days later, but the descriptions differ (overlap 0.09).

- **Three opinion-type values occur among the confirmed:** No Objection (OBS only),
  Closed, Comment Issued. So "Closed" can also be a consultation response.
- **Most candidates stay unconfirmed:** 282 have no numbered street address, and 99 with
  one have no strict match. They stay loaded (decided with the user: only confirmed ids
  are dropped).

## Step 5 · DC/21/122666 ("FOOB", 120 Fleet Street EC4A)

Found in the description notebook (Step 3): description "FOOB", decision Closed
20/07/2021, site 120 Fleet Street, London EC4A 2BE. Look for records at that address in
other authorities: first the City of London within a year either side, then any authority
at any date (read-only, approved exception).

In [ ]:
def at_address(phrase: str, authority: str | None, window: dict | None) -> pd.DataFrame:
    flt = [{"match_phrase": {"site_name": phrase}}]
    if authority:
        flt.append({"term": {"lpa_name.raw": authority}})
    if window:
        flt.append(window)
    body = {"size": 20, "_source": ["id", "lpa_name", "lpa_app_no", "decision", "decision_date", "site_name", "description"],
            "query": {"bool": {"filter": flt, "must_not": [{"term": {"lpa_name.raw": "Lewisham"}}]}}}
    found = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]],
                         columns=["id", "lpa_name", "lpa_app_no", "decision", "decision_date", "site_name", "description"])
    found["site_name"] = found["site_name"].fillna("").str.replace(r"[\r\n]+", " ", regex=True).str.slice(0, 50)
    found["description"] = found["description"].fillna("").str.slice(0, 100)
    return found


d = pd.to_datetime("20/07/2021", format="%d/%m/%Y")
year = {"range": {"decision_date": {"gte": (d - pd.Timedelta(days=365)).strftime("%d/%m/%Y"),
                                    "lte": (d + pd.Timedelta(days=365)).strftime("%d/%m/%Y"), "format": "dd/MM/yyyy"}}}
pd.set_option("display.max_colwidth", 100); pd.set_option("display.width", 300)
for phrase in ["120 FLEET STREET", "EC4A 2BE"]:
    print(f"City of London, {phrase}, within a year:", len(at_address(phrase, "City of London", year)))
    anywhere = at_address(phrase, None, None)
    print(f"Any authority, {phrase}, any date: {len(anywhere)}")
    if len(anywhere):
        print(anywhere.drop(columns=["id"]).to_string(index=False))
    print()

In [ ]:
# Southwark's record names the City's proposal: 21/00538/FULEIA. Look it up.
body = {"size": 5, "_source": ["lpa_name", "lpa_app_no", "decision", "decision_date", "site_name", "description"],
        "query": {"bool": {"filter": [{"term": {"lpa_name.raw": "City of London"}},
                                      {"match_phrase": {"lpa_app_no": "21/00538/FULEIA"}}]}}}
found = [h["_source"] for h in search(body)["hits"]["hits"]]
print(len(found), "record(s)")
for f in found:
    print({k: (v[:120] if isinstance(v, str) else v) for k, v in f.items()})

### Findings · Step 5

**Confirmed: DC/21/122666 is a consultation response.**
- The City of London's application is **21/00538/FULEIA** ("Demolition of the existing
  River Court building at 120 Fleet Street…"), Approved 20/05/2022. Its site_name is
  empty, so the address search did not find it; Southwark's record names it.
- **Five other authorities have the same kind of record** for 120 Fleet Street, all in
  summer 2021: Westminster 21/04817/OBS (No Objection (OBS only), 30/07/2021), Tower Hamlets
  PA/21/01639/NC (No Objection (OBS only), 02/08/2021), Southwark 21/OB/0047 (Closed,
  22/09/2021: "Request for observations from City of London in respect of proposal
  21/00538/FULEIA"), Camden and Hackney (no decision).
- Lewisham's DC/21/122666 (Closed, 20/07/2021, description "FOOB") is Lewisham's record of
  the same consultation.
- **A pattern:** when a large scheme is consulted on widely, each consulted borough gets
  its own record at the other borough's address. "Closed" is one of the values used.